# Jarvis wake phrases: training for openWakeWord

This trains one openWakeWord model for each of Jarvis's missing default phrases:
**Hello Jarvis**, **Wake up Jarvis** and **Jarvis wake up**. ("Hey Jarvis" already has a model.)

**How to run it**
1. At the top, choose **Runtime → Change runtime type → T4 GPU**, then Save.
2. Choose **Runtime → Run all**.
3. Leave the tab open. It takes a while: about 1 hour per phrase, according to openWakeWord's
   own documentation (not measured by Jarvis). Downloading the training data (~17 GB) happens inside Google's
   computer, not yours.
4. At the end, your browser downloads **jarvis_wake_models.zip**. Give that file to Claude.

If a cell stops with a red error, copy the error text and send it to Claude.

This is openWakeWord's official automatic training notebook with three changes: it trains a list of
phrases, it skips the TensorFlow Lite copy (Jarvis uses the ONNX model), and it packs the results into one download.

In [ ]:
## Phrases to train. Add or remove lines here to train others later.
PHRASES = ["hello jarvis", "wake up jarvis", "jarvis wake up"]

# Setup

In [ ]:
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize webrtcvad

!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19 onnx

# Jarvis only needs the ONNX model: skip the TensorFlow Lite conversion (its old TensorFlow does not install on today's Colab).
p = "openwakeword/openwakeword/train.py"
t = open(p).read()
t = t.replace("        convert_onnx_to_tflite(", "        if False: convert_onnx_to_tflite(")
open(p, "w").write(t)

import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
for f in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]:
    os.system(f"wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f} -O ./openwakeword/openwakeword/resources/models/{f}")

In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download training data

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Train each phrase

In [ ]:
import subprocess
for phrase in PHRASES:
    config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
    config["target_phrase"] = [phrase]
    config["model_name"] = phrase.replace(" ", "_")
    config["output_dir"] = "./jarvis_models"
    config["background_paths"] = ["./audioset_16k", "./fma"]
    config["false_positive_validation_data_path"] = "validation_set_features.npy"
    config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
    cfg = f"{config['model_name']}.yaml"
    yaml.dump(config, open(cfg, "w"))
    for step in ["--generate_clips", "--augment_clips", "--train_model"]:
        print(f"=== {phrase}: {step} ===", flush=True)
        subprocess.run([sys.executable, "openwakeword/openwakeword/train.py", "--training_config", cfg, step], check=True)

# Download the models

In [ ]:
import glob, zipfile
from google.colab import files
models = glob.glob("jarvis_models/*.onnx")
print("Trained:", models)
with zipfile.ZipFile("jarvis_wake_models.zip", "w") as z:
    for m in models:
        z.write(m, os.path.basename(m))
files.download("jarvis_wake_models.zip")